# BAB IV - CRISP-DM: Klasifikasi Software Inventory GLPI

**VS Code / Jupyter | Input: `RawData.csv` (atau `RawData.xlsx`) | Variabel: `RawData`**

**VERSI MANDIRI:** Fungsi `glpi_shared` sudah dimasukkan langsung ke notebook, sehingga tidak perlu file `glpi_shared.py` untuk menjalankan notebook ini. Dashboard Streamlit tetap membutuhkan file `glpi_shared.py` yang tersedia dalam ZIP proyek.

Dua kelas: `INCLUDE` dan `EXCLUDE` berdasarkan sembilan keluarga software dari tim IT. `REVIEW` hanyalah status pemeriksaan.

**Validasi:** Label awal berbasis aturan; metrik sebelum verifikasi ahli hanya mengukur kesesuaian terhadap aturan.

Seluruh output BAB IV tersimpan di `Hasil/`.


## 4.1 Business Understanding

### Kode 01 — Persiapan library dan lokasi proyek

In [ ]:
# Kode 01: library + fungsi GLPI (mandiri, tanpa file glpi_shared.py)
"""Shared GLPI IO, nine-family draft rules and safe exports.

The nine families are supplied by the IT supervisor. Draft labels produced
by classify() are not independently verified ground truth.
"""
import io, re, csv, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

FAMILIES = [
    'Adobe Acrobat', 'Autodesk AutoCAD', 'Microsoft Office',
    'Microsoft Visio', 'Microsoft Windows', 'IZArc', 'SketchUp',
    'Adobe Photoshop', 'Adobe Illustrator'
]

# Deterministic rules for a FIRST PASS; no learning model is used.
# Matching uses temporary lowercasing; source name remains exactly as imported.
RE_SUPPORT = re.compile(
    r'\b(?:language\s+packs?|font\s+packs?|fonts?\s+support|mui|proof(?:ing)?|'
    r'service\s+packs?|hotfix\s*\d*|updates?|patch(?:es)?|'
    r'extensions?|plugins?|plug\s*-?ins?|add\s*-?ons?|'
    r'components?|redistributables?|runtime|sdk|api|'
    r'object\s+enabler|vba\s+enabler|enabler|metadata|'
    r'click\s*-?\s*to\s*-?\s*run|'
    r'licensing\s+component|localization\s+component|'
    r'extensibility\s+component|driver\s+package|setup\s+metadata|'
    r'import\s*sdk|importer|import|\bservice\b|shared\s+language|'
    r'content\s+pack|private\s+pack|resources?|feedback|\bhelp\b|'
    r'\bcore\b|\bshared\b|\bprivate\b|security\s+fix|'
    r'developer\s+tools?|targeting|notification\s+client|open\s+in\s+desktop)\b',
    re.I
)
RE_WIN_EDITION = re.compile(
    r'^(?:microsoft\s+)?windows\s+(?:(?:xp|vista|7|8(?:\.1)?|10|11|2000|nt|95|98|me)\b|'
    r'(?:storage\s+)?server\s+(?:\d{4}|\d{2}|standard|datacenter|essentials|enterprise)\b|'
    r'(?:pro|professional|enterprise|home|education|ultimate|business|embedded|iot)\b)', re.I
)
RE_OFFICE_MAIN = re.compile(
    r'^(?:microsoft\s+|ms\s+)?office(?:\s+|$)|^microsoft\s+365\s*(?:\(|apps?\b|$)',re.I
)
RE_OFFICE_ACCESSORY = re.compile(
    r'\b(?:project|sharepoint|infopath|lync|groove|sway|copilot|remote|'
    r'actionsserver|file\s+validation|database\s+engine|'
    r'proof|shared|click\s*-?to\s*-?run)\b',re.I
)
RE_NON_MAIN_WIN = re.compile(
    r'\b(?:installation\s+assistant|update\s+assistant|xp\s+mode|'
    r'desktop\s+runtime|app\s+runtime|sdk|driver|service|'
    r'subsystem|feature|language|tool|helper|app|pack|'
    r'movie\s+maker|media\s+player|defender)\b',re.I
)
RE_AFTER_VISIO = re.compile(r'\b(?:viewer|mvp\s+sessions)\b',re.I)


def classify(name: str):
    n = re.sub(r'\s+', ' ', name).strip()
    l = n.casefold()
    if not n:
        return 'EXCLUDE', '', 'Nama software kosong', 'CEK_NAMA_KOSONG'
    # Visio is sometimes stored as Microsoft Office Visio; test before Office.
    if re.search(r'\bvisio\b', l) and re.search(r'\bvisio\b', l):
        if (re.search(r'^(?:microsoft\s+)?(?:office\s+)?visio\b',l)
             or re.search(r'^visio\s+(?:\d{4}|ltd|l t s c)',l)):
            if RE_SUPPORT.search(l) or RE_AFTER_VISIO.search(l):
                return 'EXCLUDE', '', 'Komponen/penampil/pembaruan Visio, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
            return 'INCLUDE', 'Microsoft Visio', 'Aplikasi Microsoft Visio (semua versi)', ''
    # Acrobat: Reader is a main end-user application, not its language fonts.
    if (re.search(r'^(?:adobe\s+)?acrobat\b',l) or re.search(r'^adobe\s+reader\b',l)):
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Komponen/pack Acrobat/Reader, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'Adobe Acrobat', 'Aplikasi Acrobat/Reader (semua versi)', ''
    # AutoCAD and named editions e.g. AutoCAD LT, Civil 3D, Plant 3D.
    if re.search(r'^(?:autodesk\s+)?auto\s?cad\b', l):
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Komponen/update/pack AutoCAD, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'Autodesk AutoCAD', 'Aplikasi AutoCAD (termasuk varian/versi utama)', ''
    if re.search(r'^adobe\s+photoshop\b',l):
        if re.search(r'^adobe\s+photoshop\s+lightroom\b',l):
            return 'EXCLUDE', '', 'Lightroom merupakan produk lain, bukan aplikasi Photoshop utama', 'CEK_MIRIP_TARGET'
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Komponen/pack Photoshop, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'Adobe Photoshop', 'Aplikasi Photoshop (semua versi)', ''
    if re.search(r'^(?:adobe|abobe)\s+illustrator\b',l):
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Komponen/pack Illustrator, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'Adobe Illustrator', 'Aplikasi Illustrator (semua versi)', ''
    if re.search(r'^sketch\s?up\b',l):
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Plugin/importer SketchUp, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'SketchUp', 'Aplikasi SketchUp (semua versi)', ''
    if re.search(r'^izarc\b',l):
        if RE_SUPPORT.search(l):
            return 'EXCLUDE', '', 'Komponen IZArc, bukan aplikasi utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'IZArc', 'Aplikasi kompresi IZArc', ''
    # Microsoft Office, Office Std, Microsoft 365 Apps (not WPS/LibreOffice)
    if RE_OFFICE_MAIN.search(l):
        if re.search(r'\bvisio\b',l):
            return 'EXCLUDE', '', 'Varian Office/Visio bukan aplikasi utama yang teridentifikasi', 'CEK_KOMPONEN_TARGET'
        if RE_SUPPORT.search(l) or RE_OFFICE_ACCESSORY.search(l):
            return 'EXCLUDE', '', 'Komponen/produk pendukung Office, bukan paket utama', 'CEK_KOMPONEN_TARGET'
        if l == 'office' or l == 'microsoft 365' or l == 'microsoft 365 (office)':
            return 'INCLUDE', 'Microsoft Office', 'Paket aplikasi Office / Microsoft 365', ''
        if (re.search(r'^(?:microsoft\s+|ms\s+)?office\s+(?:\d{2,4}|std|standard|professional|pro|enterprise|home|business|student|ltd|ltsc|access|excel|word|powerpoint|publisher|onenote|outlook|single|premium)\b',l)
            or re.search(r'^microsoft\s+365\s+apps?\b',l)):
            return 'INCLUDE', 'Microsoft Office', 'Aplikasi/paket Microsoft Office (semua versi)', ''
        return 'EXCLUDE', '', 'Nama berkaitan Office tetapi tidak teridentifikasi sebagai paket utama', 'CEK_MIRIP_TARGET'
    # Windows - avoid arbitrary third-party software "for Windows".
    if RE_WIN_EDITION.search(l):
        if RE_SUPPORT.search(l) or RE_NON_MAIN_WIN.search(l):
            return 'EXCLUDE', '', 'Komponen/update/utility Windows, bukan OS utama', 'CEK_KOMPONEN_TARGET'
        return 'INCLUDE', 'Microsoft Windows', 'Sistem operasi Windows (semua versi)', ''
    # Default from the IT policy: everything outside these families is EXCLUDE.
    if re.search(r'\b(?:acrobat|autocad|auto\s+cad|photoshop|illustrator|visio|sketchup|sketch\s+up|izarc)\b',l):
        return 'EXCLUDE', '', 'Di luar sembilan program utama atau hanya menyebut produk', 'CEK_MIRIP_TARGET'
    if re.search(r'^(?:adobe\s+pro\b|microsoft\s+windows\b)',l):
        return 'EXCLUDE', '', 'Nama berpotensi ambigu; perlu cek sebagai produk utama', 'CEK_MIRIP_TARGET'
    return 'EXCLUDE', '', 'Di luar sembilan keluarga software yang ditetapkan IT', ''



def decode_mixed(raw: bytes) -> str:
    """Keep valid UTF-8, decode isolated Windows bytes as cp1252. No silent row skips."""
    text = raw.decode('utf-8-sig', errors='surrogateescape')
    chars = []
    for char in text:
        code = ord(char)
        if 0xDC80 <= code <= 0xDCFF:
            chars.append(bytes([code - 0xDC00]).decode('cp1252', errors='replace'))
        else:
            chars.append(char)
    return ''.join(chars)

def read_table(path):
    """Accept CSV, tab-delimited exports with .csv suffix, or Excel."""
    path = Path(path)
    if path.suffix.lower() in ('.xlsx','.xls'):
        frame = pd.read_excel(path, dtype=str, keep_default_na=False)
    elif path.suffix.lower() in ('.csv','.tsv','.txt'):
        content = decode_mixed(path.read_bytes())
        header = content.splitlines()[0].lstrip('\ufeff')
        choices = [',','\t',';']
        delimiter = max(choices, key=lambda d: header.count(d))
        if header.count(delimiter) == 0:
            delimiter = ','
        try:
            frame = pd.read_csv(io.StringIO(content), sep=delimiter, dtype=str,
                                keep_default_na=False, engine='python', on_bad_lines='error')
        except Exception as exc:
            raise ValueError(f'File {path.name} sulit dibaca. Periksa quote/pemisah. Detail: {exc}') from exc
    else:
        raise ValueError('File harus .csv, .tsv, .txt, atau .xlsx')
    frame.columns = [str(c).strip().lstrip('\ufeff') for c in frame.columns]
    if 'name' not in frame.columns:
        raise ValueError(f'Kolom name wajib ada. Kolom tersedia: {list(frame.columns)}')
    if 'id' not in frame.columns:
        frame.insert(0, 'id', [str(i+1) for i in range(len(frame))])
    return frame.fillna('').astype(str)

def clean_name(text):
    s = unicodedata.normalize('NFKC',str(text)).casefold()
    s = re.sub(r'[,;|]+',' ',s)
    s = re.sub(r'\s+',' ',s).strip()
    return s

def label_frame(raw):
    df = raw.copy()
    if 'label' in df.columns:
        df['label_dari_file_awal'] = df['label']
    labels = [classify(n) for n in df['name'].fillna('')]
    df['label'] = [t[0] for t in labels]
    df['keluarga_include'] = [t[1] for t in labels]
    df['dasar_label'] = [t[2] for t in labels]
    df['penanda_pemeriksaan'] = [t[3] for t in labels]
    df['nama_bersih'] = df['name'].map(clean_name)
    df['status_label'] = np.where(df['penanda_pemeriksaan'].astype(bool), 'PERLU_REVIEW', 'LABEL_AWAL_ATURAN')
    return df

def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')

def excel_cell(v):
    if v is None:
        return ''
    if not isinstance(v,str):
        return v
    v = ''.join(c for c in v if ord(c)>=32 or c in '\t\n\r')
    if v.startswith(('=', '+', '-', '@')):
        return "'"+v
    return v

def write_excel(tables, path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    wb = Workbook()
    wb.remove(wb.active)
    for name, df in tables.items():
        ws = wb.create_sheet(name[:31])
        ws.append(list(df.columns))
        for cell in ws[1]:
            cell.fill = PatternFill('solid',fgColor='183153')
            cell.font = Font(name='Aptos',color='FFFFFF',bold=True)
            cell.alignment = Alignment(vertical='center',wrap_text=True)
        ws.row_dimensions[1].height = 28
        for row in df.itertuples(index=False,name=None):
            ws.append([excel_cell(x) for x in row])
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False
        for i,col in enumerate(df.columns,1):
            ws.column_dimensions[get_column_letter(i)].width = 49 if col in ['name','nama_bersih','dasar_label'] else min(32,max(14,len(col)+2))
    wb.save(path)
    return path


# Persiapan penelitian dan folder output
from pathlib import Path
import io, json, re, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
import joblib
# Fungsi GLPI sudah didefinisikan di atas.

BASE = Path.cwd()
HASIL = BASE / 'Hasil'
FOLDERS = {
    'business': HASIL / '01_Business_Understanding',
    'understanding': HASIL / '02_Data_Understanding',
    'preparation': HASIL / '03_Data_Preparation',
    'modeling': HASIL / '04_Modeling',
    'evaluation': HASIL / '05_Evaluation',
    'deployment': HASIL / '06_Deployment',
    'narasi': HASIL / '07_Narasi_BAB_IV',
}
for folder in FOLDERS.values(): folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'figure.facecolor':'white','axes.grid':False})

def save_fig(folder, filename):
    dest = FOLDERS[folder] / filename
    plt.tight_layout()
    plt.savefig(dest, dpi=180, bbox_inches='tight', facecolor='white')
    plt.close()
    display(Image(filename=str(dest),width=850))
    return dest

print('VS Code working directory :', BASE)
print('Folder Hasil              :', HASIL)

### Kode 02 — Identifikasi tujuan dan kategori keluaran

In [ ]:
tujuan = pd.DataFrame([
  ['Masalah bisnis','Software Inventory memuat aplikasi utama dan komponen pendukung yang bercampur'],
  ['Target klasifikasi','INCLUDE atau EXCLUDE berdasarkan 9 keluarga software prioritas tim IT'],
  ['Kelas 1','INCLUDE: aplikasi utama dari keluarga software yang ditetapkan IT'],
  ['Kelas 0','EXCLUDE: software lain di luar kebutuhan utama saat ini'],
  ['Status kerja','REVIEW: nama ambigu atau ketiga model berbeda; bukan kelas ML'],
  ['Pemanfaatan','Daftar rekomendasi dan dashboard; tidak mengubah GLPI produksi'],
], columns=['Aspek','Penjelasan'])
write_csv(tujuan,FOLDERS['business']/'Tabel_4_1_Tujuan_Penelitian.csv')
display(tujuan)

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(13.2,2.9))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
phases=['Business\nUnderstanding','Data\nUnderstanding','Data\nPreparation','Modeling','Evaluation','Deployment']
for i,phase in enumerate(phases):
    x=.01+i*.1665
    box=FancyBboxPatch((x,.35),.145,.31,boxstyle='round,pad=.012,rounding_size=.025',
                       facecolor='#e9f2fa',edgecolor='#1c5775',linewidth=1.4)
    ax.add_patch(box)
    ax.text(x+.0725,.5,phase,ha='center',va='center',fontsize=9.5,fontweight='bold')
    if i<5:
        ax.add_patch(FancyArrowPatch((x+.156,.5),(x+.164,.5),
                        arrowstyle='-|>',mutation_scale=13,linewidth=1.4,color='#376779'))
ax.set_title('Gambar 4.1 - Alur CRISP-DM Penelitian GLPI',fontsize=13,pad=8)
save_fig('business','Gambar_4_1_Alur_CRISP_DM.png')

## 4.2 Data Understanding

### Kode 03 — Membaca dataset RawData (CSV/TSV/Excel)

In [1]:
input_files=[BASE/'RawData.csv',BASE/'RawData.xlsx',BASE/'RawData.xls',BASE/'RawData.tsv']
input_path=next((p for p in input_files if p.exists()),None)
if input_path is None:
    raise FileNotFoundError('Letakkan RawData.csv atau RawData.xlsx di folder proyek VS Code.')
RawData=read_table(input_path)
print('File   :', input_path.name)
print('Baris  :', len(RawData))
print('Kolom  :', len(RawData.columns))
print('Kolom  :', list(RawData.columns))
display(RawData.head(8))

NameError: name 'BASE' is not defined

### Kode 04 — Deskripsi dataset dan nilai kosong

In [ ]:
summary=pd.DataFrame([
    ['Total record',len(RawData)],['Total kolom',RawData.shape[1]],
    ['Nama kosong',int(RawData['name'].str.strip().eq('').sum())],
    ['Nama unik persis',RawData['name'].nunique()],
    ['ID unik',RawData['id'].nunique()],
],columns=['Metrik','Nilai'])
missing=(RawData.astype(str).apply(lambda s:s.str.strip().eq('').sum())
         .sort_values(ascending=False).rename_axis('Kolom').reset_index(name='Jumlah_Kosong'))
missing['Persentase']=100*missing['Jumlah_Kosong']/len(RawData)
write_csv(summary,FOLDERS['understanding']/'Tabel_4_2_Ringkasan_Data.csv')
write_csv(missing,FOLDERS['understanding']/'Tabel_4_3_Nilai_Kosong.csv')
display(summary);display(missing.head(12))
fig,ax=plt.subplots(figsize=(11,5))
top=missing.head(13).sort_values('Jumlah_Kosong')
ax.barh(top['Kolom'],top['Jumlah_Kosong'],color='#648f9f')
ax.set(xlabel='Jumlah nilai kosong',title='Gambar 4.2 - Pemeriksaan Missing Values')
save_fig('understanding','Gambar_4_2_Missing_Values.png')

### Kode 05 — Distribusi kategori dan manufacturer dari GLPI

In [ ]:
for field,number,title,filename in [
 ('software_category_name',3,'Distribusi kategori software GLPI','Gambar_4_3_Kategori_GLPI.png'),
 ('manufacturer_name',4,'Distribusi manufacturer GLPI','Gambar_4_4_Manufacturer_GLPI.png')]:
    if field not in RawData.columns:continue
    series=RawData[field].replace('','(Tidak diisi)').value_counts().head(12)
    tab=series.rename_axis(field).reset_index(name='Jumlah')
    write_csv(tab,FOLDERS['understanding']/f'Tabel_4_{number+1}_{field}.csv')
    fig,ax=plt.subplots(figsize=(11,5))
    ax.barh(series.index[::-1],series.values[::-1],color='#508c8b')
    ax.set(xlabel='Jumlah record',title=f'Gambar 4.{number} - {title}')
    save_fig('understanding',filename)

### Kode 06 — Analisis nama berulang dan panjang teks

In [ ]:
RawData['nama_grup_eda']=RawData['name'].map(clean_name)
freq=RawData.loc[RawData['nama_grup_eda'].ne(''),'nama_grup_eda'].value_counts()
ulang=int((RawData['nama_grup_eda'].ne('') & RawData['nama_grup_eda'].duplicated(keep=False)).sum())
stat_duplicates=pd.DataFrame([['Record dengan nama berulang',ulang],
                              ['Nama unik setelah case folding dan spasi',len(freq)],
                              ['Nama yang muncul lebih dari sekali',int((freq>1).sum())]],columns=['Metrik','Nilai'])
write_csv(stat_duplicates,FOLDERS['understanding']/'Tabel_4_6_Duplikasi_Nama.csv')
write_csv(freq.head(60).rename_axis('nama').reset_index(name='frekuensi'),
          FOLDERS['understanding']/'Tabel_4_7_Top_Nama_Berulang.csv')
display(stat_duplicates)
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].bar(['Nama hanya sekali','Record nama berulang'],[len(RawData)-ulang,ulang],color=['#6ca69a','#d9a85b'])
axs[0].set_title('Gambar 4.5 - Rekaman nama berulang')
axs[1].hist(RawData['name'].str.len(),bins=30,color='#578ea0')
axs[1].set(xlabel='Panjang karakter',ylabel='Frekuensi',title='Gambar 4.6 - Panjang nama software')
save_fig('understanding','Gambar_4_5_6_Duplikasi_Panjang.png')

## 4.3 Data Preparation

### Kode 07 — Preprocessing teks nama software (tanpa menghapus data mentah)

In [ ]:
Data_Preparation=RawData.copy()
Data_Preparation['nama_bersih']=Data_Preparation['name'].map(clean_name)
contoh=Data_Preparation[['id','name','nama_bersih']].head(25)
write_csv(contoh,FOLDERS['preparation']/'Tabel_4_8_Sebelum_Sesudah_Preprocessing.csv')
stat_pre=pd.DataFrame([
 ['Record asli',len(RawData)],
 ['Nama kosong sebelum cleaning',int(RawData['name'].str.strip().eq('').sum())],
 ['Nama kosong setelah cleaning',int(Data_Preparation['nama_bersih'].eq('').sum())],
 ['Nama unik sebelum cleaning',RawData['name'].nunique()],
 ['Nama unik setelah cleaning',Data_Preparation['nama_bersih'].nunique()],
],columns=['Tahap','Jumlah'])
write_csv(stat_pre,FOLDERS['preparation']/'Tabel_4_9_Ringkasan_Preprocessing.csv')
display(stat_pre);display(contoh.head(10))
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(['Sebelum','Sesudah'],[RawData['name'].nunique(),Data_Preparation['nama_bersih'].nunique()],color='#688ea2')
ax.set(ylabel='Jumlah nama unik',title='Gambar 4.7 - Perbandingan nama unik sebelum/sesudah preprocessing')
save_fig('preparation','Gambar_4_7_Preprocessing.png')

### Kode 08 — Labeling dua kelas berdasarkan 9 keluarga prioritas Pak Widi

In [ ]:
Data_Berlabel=label_frame(RawData)
assert set(Data_Berlabel['label']) <= {'INCLUDE','EXCLUDE'}
assert len(Data_Berlabel)==len(RawData)
write_csv(Data_Berlabel,FOLDERS['preparation']/'Dataset_Berlabel_Awal.csv')
label_counts=Data_Berlabel['label'].value_counts().reindex(['INCLUDE','EXCLUDE'],fill_value=0)
family_counts=(Data_Berlabel.loc[Data_Berlabel['label'].eq('INCLUDE'),'keluarga_include']
               .value_counts().reindex(FAMILIES,fill_value=0))
write_csv(label_counts.rename_axis('Label').reset_index(name='Jumlah'),FOLDERS['preparation']/'Tabel_4_10_Distribusi_Label.csv')
write_csv(family_counts.rename_axis('Keluarga').reset_index(name='Jumlah'),FOLDERS['preparation']/'Tabel_4_11_Keluarga_INCLUDE.csv')
print('Label berdasarkan aturan awal, belum terverifikasi:')
print(label_counts.to_string())
fig,axs=plt.subplots(1,2,figsize=(13,5))
axs[0].bar(label_counts.index,label_counts.values,color=['#2f9d80','#b6bbc7']);axs[0].set_title('Gambar 4.8 - Distribusi label awal')
axs[1].barh(family_counts.index[::-1],family_counts.values[::-1],color='#438da0')
axs[1].set_title('Gambar 4.9 - Keluarga software INCLUDE')
save_fig('preparation','Gambar_4_8_9_Distribusi_Label_dan_Keluarga.png')

### Kode 09 — Daftar nama ambigu untuk pemeriksaan IT

In [ ]:
review_candidates=Data_Berlabel[Data_Berlabel['penanda_pemeriksaan'].ne('')].copy()
write_csv(review_candidates,FOLDERS['preparation']/'Daftar_Perlu_Pemeriksaan.csv')
review_summary=Data_Berlabel['status_label'].value_counts().rename_axis('Status').reset_index(name='Jumlah')
write_csv(review_summary,FOLDERS['preparation']/'Tabel_4_12_Status_Label_Awal.csv')
print('Kasus bertanda PERLU_REVIEW:',len(review_candidates))
display(review_candidates[['id','name','label','penanda_pemeriksaan']].head(12))

### Kode 10 — Template validasi supervisor (tidak mengklaim label sudah divalidasi)

In [ ]:
# Pilih nama unik: sampel acak seimbang + kasus ambigu prioritas.
valid_pool=Data_Berlabel.loc[Data_Berlabel['nama_bersih'].ne('')].drop_duplicates('nama_bersih').copy()
ambig=valid_pool[valid_pool['penanda_pemeriksaan'].ne('')].sample(
    n=min(140,int(valid_pool['penanda_pemeriksaan'].ne('').sum())),random_state=42)
others=valid_pool.drop(index=ambig.index)
chosen=[ambig]
for lab in ['INCLUDE','EXCLUDE']:
    candidate=others[others['label'].eq(lab)]
    chosen.append(candidate.sample(n=min(150,len(candidate)),random_state=42))
valid_template=pd.concat(chosen,ignore_index=True).drop_duplicates('nama_bersih')
valid_template=valid_template[['id','name','label','keluarga_include','penanda_pemeriksaan']].rename(columns={'label':'label_awal'})
valid_template['label_verifikasi']=''
valid_template['catatan_supervisor']=''
write_csv(valid_template,FOLDERS['preparation']/'Template_Validasi_Pak_Widi.csv')
write_excel({'Validasi':valid_template},FOLDERS['preparation']/'Template_Validasi_Pak_Widi.xlsx')
print('Template validasi:',len(valid_template),'nama unik. Isi label_verifikasi lalu simpan sebagai Label_Terverifikasi.xlsx di folder proyek.')

## 4.4 Modeling

### Kode 11 — Menentukan sumber label uji: verified atau label aturan awal

In [ ]:
# Kalau tersedia, pakai label_verifikasi dari Pak Widi untuk evaluasi terpisah.
# Kalau belum, split label otomatis pada NAMA UNIK (bukan pada baris yang berulang).
all_unique=(Data_Berlabel.loc[Data_Berlabel['nama_bersih'].ne('')]
            .drop_duplicates('nama_bersih').copy())
train_pool=all_unique.loc[all_unique['penanda_pemeriksaan'].eq('')].copy()
verified_path=next((p for p in [BASE/'Label_Terverifikasi.xlsx',BASE/'Label_Terverifikasi.csv'] if p.exists()),None)
verified=None
if verified_path is not None:
    v=read_table(verified_path)
    if 'label_verifikasi' not in v.columns:
        raise ValueError('Label_Terverifikasi harus memiliki kolom label_verifikasi (INCLUDE/EXCLUDE).')
    v['label_verifikasi']=v['label_verifikasi'].str.upper().str.strip().replace({'EXCLUDE_CANDIDATE':'EXCLUDE'})
    v=v[v['label_verifikasi'].isin(['INCLUDE','EXCLUDE'])].copy()
    v['nama_bersih']=v['name'].map(clean_name)
    conflicts=(v.groupby('nama_bersih')['label_verifikasi'].nunique())
    conflict_keys=set(conflicts[conflicts>1].index)
    if conflict_keys:
        write_csv(v[v['nama_bersih'].isin(conflict_keys)],FOLDERS['preparation']/'Konflik_Label_Verifikasi.csv')
        v=v[~v['nama_bersih'].isin(conflict_keys)]
    verified=v.drop_duplicates('nama_bersih').copy()
verified_ok=(verified is not None and len(verified)>=30 and verified['label_verifikasi'].value_counts().min()>=5 and verified['label_verifikasi'].nunique()==2)
if verified_ok:
    test_df=verified[['id','name','nama_bersih','label_verifikasi']].rename(columns={'label_verifikasi':'label'}).copy()
    train_df=train_pool[~train_pool['nama_bersih'].isin(set(test_df['nama_bersih']))].copy()
    evaluation_status='VERIFIKASI_SUPERVISOR'
else:
    train_df,test_df=train_test_split(train_pool,test_size=.20,random_state=42,stratify=train_pool['label'])
    train_df=train_df.copy(); test_df=test_df.copy()
    evaluation_status='KESESUAIAN_LABEL_ATURAN_BELUM_GROUND_TRUTH'
assert set(train_df['nama_bersih']).isdisjoint(set(test_df['nama_bersih']))
assert train_df['label'].nunique()==2 and test_df['label'].nunique()==2
print('Mode evaluasi:',evaluation_status)
print('Data train:',len(train_df),'Data test:',len(test_df))
print('Distribusi test:',test_df['label'].value_counts().to_dict())
write_csv(train_df,FOLDERS['modeling']/'Data_Train_Nama_Unik.csv')
write_csv(test_df,FOLDERS['modeling']/'Data_Test_Nama_Unik.csv')
split=pd.concat([train_df['label'].value_counts().rename('Train'),test_df['label'].value_counts().rename('Test')],axis=1).fillna(0).astype(int)
write_csv(split.rename_axis('Label').reset_index(),FOLDERS['modeling']/'Tabel_4_13_Distribusi_Train_Test.csv')
fig,ax=plt.subplots(figsize=(8,4))
split.T.plot(kind='bar',stacked=True,ax=ax,color=['#4da58d','#9fb7bd'])
ax.set(ylabel='Jumlah nama unik',xlabel='Subset',title='Gambar 4.10 - Distribusi data train dan test')
save_fig('modeling','Gambar_4_10_Train_Test.png')

### Kode 12 — Konfigurasi TF-IDF dan tiga algoritma klasifikasi

In [ ]:
# Char n-gram cocok untuk variasi nama software, versi, dan singkatan.
# Hanya kolom nama_bersih yang dipakai sebagai fitur; tidak memakai label, keluarga, kategori, atau flag.
def vectorizer():
    return TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,
                           max_features=25000,sublinear_tf=True)
model_specs={
    'Logistic_Regression':LogisticRegression(max_iter=1800,class_weight='balanced',random_state=42),
    'Linear_SVM':LinearSVC(class_weight='balanced',random_state=42),
    'Multinomial_Naive_Bayes':MultinomialNB(alpha=.5),
}
models={name:Pipeline([('tfidf',vectorizer()),('model',classifier)])
        for name,classifier in model_specs.items()}
X_train=train_df['nama_bersih'].to_numpy()
y_train=train_df['label'].to_numpy()
X_test=test_df['nama_bersih'].to_numpy()
y_test=test_df['label'].to_numpy()
print('Tiga model:',list(models))

### Kode 13 — Training dan penyimpanan ketiga model

In [ ]:
for name,model in models.items():
    model.fit(X_train,y_train)
    joblib.dump(model,FOLDERS['modeling']/f'Model_{name}.joblib')
    print(name,'- fitur TF-IDF:',len(model.named_steps['tfidf'].get_feature_names_out()))
example_vector=models['Linear_SVM'].named_steps['tfidf'].transform(X_train[:3])
features=models['Linear_SVM'].named_steps['tfidf'].get_feature_names_out()
rows=[]
for i in range(example_vector.shape[0]):
    idx=example_vector[i].nonzero()[1]
    for j in sorted(idx,key=lambda v:-float(example_vector[i, v]))[:8]:
        rows.append({'Nama':X_train[i],'Fitur_TFIDF':features[j],'Bobot':round(float(example_vector[i,j]),5)})
write_csv(pd.DataFrame(rows),FOLDERS['modeling']/'Tabel_4_14_Contoh_TFIDF.csv')

### Kode 14 — Cross-validation tiga model pada data latih

In [ ]:
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
cv_rows=[]
for name,model in models.items():
    scores=cross_validate(model,X_train,y_train,cv=cv,scoring={
      'f1_include':'f1_macro','balanced_accuracy':'balanced_accuracy'},n_jobs=1)
    cv_rows.append({'Model':name,'CV_F1_Macro_Mean':round(scores['test_f1_include'].mean(),4),
                    'CV_F1_Macro_Std':round(scores['test_f1_include'].std(),4),
                    'CV_Balanced_Accuracy':round(scores['test_balanced_accuracy'].mean(),4)})
cv_table=pd.DataFrame(cv_rows)
write_csv(cv_table,FOLDERS['modeling']/'Tabel_4_15_Cross_Validation.csv')
display(cv_table)

## 4.5 Evaluation

### Kode 15 — Metrik evaluasi, classification report dan confusion matrix

In [ ]:
metrics_rows=[]
all_preds={}
for name,model in models.items():
    pred=model.predict(X_test)
    all_preds[name]=pred
    p,r,f,_=precision_recall_fscore_support(y_test,pred,labels=['INCLUDE'],zero_division=0)
    metrics_rows.append({
        'Model':name,'Accuracy':accuracy_score(y_test,pred),
        'Balanced_Accuracy':balanced_accuracy_score(y_test,pred),
        'Precision_INCLUDE':p[0],'Recall_INCLUDE':r[0],'F1_INCLUDE':f[0],
        'F1_Macro':precision_recall_fscore_support(y_test,pred,average='macro',zero_division=0)[2],
        'Status_Evaluasi':evaluation_status})
    write_csv(pd.DataFrame(classification_report(y_test,pred,output_dict=True,zero_division=0)).T,
              FOLDERS['evaluation']/f'Classification_Report_{name}.csv')
    cm=confusion_matrix(y_test,pred,labels=['INCLUDE','EXCLUDE'])
    write_csv(pd.DataFrame(cm,index=['Aktual_INCLUDE','Aktual_EXCLUDE'],columns=['Pred_INCLUDE','Pred_EXCLUDE']),
              FOLDERS['evaluation']/f'Confusion_Matrix_{name}.csv')
    fig,ax=plt.subplots(figsize=(5.5,4.5))
    ConfusionMatrixDisplay(cm,display_labels=['INCLUDE','EXCLUDE']).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'Gambar 4.11 - Confusion Matrix {name.replace("_"," ")}')
    save_fig('evaluation',f'Gambar_4_11_CM_{name}.png')
metrics_df=pd.DataFrame(metrics_rows)
write_csv(metrics_df,FOLDERS['evaluation']/'Tabel_4_16_Perbandingan_Kinerja_3_Model.csv')
display(metrics_df.round(4))

### Kode 16 — Visualisasi perbandingan tiga algoritma

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
fields=['Balanced_Accuracy','Precision_INCLUDE','Recall_INCLUDE','F1_INCLUDE','F1_Macro']
x=np.arange(len(fields)); width=.24
for i,model in enumerate(metrics_df['Model']):
    values=metrics_df.loc[metrics_df['Model'].eq(model),fields].values[0].astype(float)
    ax.bar(x+(i-1)*width,values,width,label=model.replace('_',' '))
ax.set_xticks(x,fields,rotation=12,ha='right')
ax.set_ylim(0,1.08);ax.set_ylabel('Skor (0-1)')
ax.set_title('Gambar 4.12 - Perbandingan performa tiga algoritma')
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.26),ncol=3,fontsize=9,frameon=False)
save_fig('evaluation','Gambar_4_12_Perbandingan_3_Model.png')

pred_test=test_df[['id','name','label']].copy()
for name,pred in all_preds.items():pred_test[f'pred_{name}']=pred
write_csv(pred_test,FOLDERS['evaluation']/'Prediksi_Pada_Data_Test.csv')
write_csv(pred_test.loc[(pred_test[[f'pred_{n}' for n in models]]!=pred_test['label'].to_numpy()[:,None]).any(axis=1)],
          FOLDERS['evaluation']/'Contoh_Kesalahan_Prediksi.csv')

### Kode 17 — Interpretasi fitur penting setiap model

In [ ]:
featrows=[]
for name,model in models.items():
    names=model.named_steps['tfidf'].get_feature_names_out()
    clf=model.named_steps['model']
    if hasattr(clf,'coef_'):
        coef=clf.coef_.ravel()
        # class 1 coef is positive, label meaning depends on sorted classes.
        include_idx=int(np.where(clf.classes_=='INCLUDE')[0][0])
        if include_idx==0:coef=-coef
    else:
        index=int(np.where(clf.classes_=='INCLUDE')[0][0])
        coef=clf.feature_log_prob_[index]-clf.feature_log_prob_[1-index]
    top=np.argsort(coef)[-12:][::-1]
    for ix in top:featrows.append({'Model':name,'Fitur':names[ix],'Skor':float(coef[ix])})
feat_df=pd.DataFrame(featrows)
write_csv(feat_df,FOLDERS['evaluation']/'Tabel_4_17_Fitur_Penting_Model.csv')
fig,axs=plt.subplots(1,3,figsize=(17,5))
for ax,(model,d) in zip(axs,feat_df.groupby('Model',sort=False)):
    d=d.head(10).iloc[::-1]
    ax.barh(d['Fitur'],d['Skor'],color='#397f90');ax.set_title(model.replace('_',' '))
plt.suptitle('Gambar 4.13 - N-gram paling terkait kelas INCLUDE',y=1.04)
save_fig('evaluation','Gambar_4_13_Fitur_Penting.png')

## 4.6 Deployment

### Kode 18 — Prediksi seluruh Software Inventory dan status REVIEW

In [ ]:
all_names=Data_Berlabel['nama_bersih'].to_numpy()
result=Data_Berlabel.copy()
pred_cols=[]
for name,model in models.items():
    col=f'prediksi_{name}'
    result[col]=model.predict(all_names)
    pred_cols.append(col)
result['prediksi_final']=result[pred_cols].mode(axis=1)[0]
result['model_sepakati']=result[pred_cols].nunique(axis=1).eq(1)
# REVIEW hanya status workflow, bukan kelas ML ke-3.
result['status_dashboard']=np.where(
    result['nama_bersih'].eq('') | result['penanda_pemeriksaan'].ne('') | (~result['model_sepakati']),
    'REVIEW',result['prediksi_final'])
result['jumlah_model_setuju']=result[pred_cols].apply(lambda row:row.value_counts().max(),axis=1)
result['status_prediksi']='EKSPERIMEN_LABEL_ATURAN' if evaluation_status.startswith('KESESUAIAN') else 'DIEVALUASI_DENGAN_LABEL_SUPERVISOR'
assert len(result)==len(RawData)
write_csv(result,FOLDERS['deployment']/'Prediksi_Seluruh_Record_3_Model.csv')
write_csv(result.loc[result['status_dashboard'].eq('REVIEW')],FOLDERS['deployment']/'Prioritas_Review_IT.csv')
write_csv(result.loc[result['status_dashboard'].eq('INCLUDE')],FOLDERS['deployment']/'Rekomendasi_INCLUDE.csv')
write_csv(result.loc[result['status_dashboard'].eq('EXCLUDE')],FOLDERS['deployment']/'Rekomendasi_EXCLUDE.csv')
counts=result['status_dashboard'].value_counts().reindex(['INCLUDE','EXCLUDE','REVIEW'],fill_value=0)
write_csv(counts.rename_axis('Status').reset_index(name='Jumlah'),FOLDERS['deployment']/'Tabel_4_18_Status_Prediksi_Dashboard.csv')
fig,ax=plt.subplots(figsize=(7.5,4.2))
ax.bar(counts.index,counts.values,color=['#2c9a7b','#8796a6','#e4a749'])
ax.set(ylabel='Jumlah record',title='Gambar 4.14 - Hasil rekomendasi dashboard')
save_fig('deployment','Gambar_4_14_Hasil_Deployment.png')
print('Status hasil akhir:',counts.to_dict())

### Kode 19 — Ekspor workbook deployment dan informasi model

In [ ]:
export_cols=['id','name','manufacturer_name','software_category_name','label','prediksi_final',
             'status_dashboard','keluarga_include','penanda_pemeriksaan','jumlah_model_setuju']
export_cols=[c for c in export_cols if c in result.columns]
export_frame=result[export_cols].copy()
write_excel({
 'Dashboard':export_frame,
 'INCLUDE':export_frame[export_frame['status_dashboard'].eq('INCLUDE')],
 'EXCLUDE':export_frame[export_frame['status_dashboard'].eq('EXCLUDE')],
 'REVIEW':export_frame[export_frame['status_dashboard'].eq('REVIEW')],
 'Kriteria_IT':pd.DataFrame({'Software_INCLUDE':FAMILIES}),
},FOLDERS['deployment']/'Dashboard_Klasifikasi_Software.xlsx')
model_meta={'mode_evaluasi':evaluation_status,'input_file':input_path.name,
            'jumlah_data_asli':int(len(RawData)), 'target_kelas':['INCLUDE','EXCLUDE'],
            'families':FAMILIES,'model_terbaik_berdasarkan_f1_include':str(metrics_df.sort_values('F1_INCLUDE',ascending=False).iloc[0]['Model'])}
(FOLDERS['deployment']/'Metadata_Model.json').write_text(json.dumps(model_meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('File siap dibaca aplikasi Streamlit: Hasil/04_Modeling/Model_*.joblib')

### Kode 20 — Narasi BAB IV berdasarkan hasil aktual dan pengarsipan

In [ ]:
metricview=metrics_df[['Model','Accuracy','Balanced_Accuracy','Precision_INCLUDE','Recall_INCLUDE','F1_INCLUDE','F1_Macro']].copy()
for col in metricview.columns[1:]:metricview[col]=metricview[col].map(lambda v:f'{v:.4f}')
metlines=[' | '.join(map(str,row)) for row in metricview.itertuples(index=False,name=None)]
report=f'''# Draf narasi BAB IV - GLPI Software Inventory\n\n## 4.1 Business Understanding\nPermasalahan yang diteliti adalah pemilahan aplikasi utama dari seluruh entri Software Inventory GLPI. Dua kelas ditetapkan sebagai INCLUDE dan EXCLUDE mengacu pada sembilan keluarga software prioritas dari tim IT. REVIEW merupakan status pemeriksaan, bukan kelas klasifikasi.\n\n## 4.2 Data Understanding\nDataset awal memuat **{len(RawData):,} record** dan **{RawData.shape[1]} kolom** dari file `{input_path.name}`. Sebanyak **{int(RawData['name'].str.strip().eq('').sum())}** record memiliki nama kosong. Terdapat **{RawData['name'].nunique()} nama unik mentah**, termasuk nilai kosong. Informasi lebih lanjut ada pada Tabel 4.2-4.7 dan Gambar 4.2-4.6.\n\n## 4.3 Data Preparation\nDilakukan case folding, penyederhanaan spasi, dan perapihan pemisah pada kolom turunan `nama_bersih`. Kolom nama asli dipertahankan. Pelabelan awal berbasis aturan kebutuhan IT menghasilkan **{int(label_counts.get('INCLUDE',0))} INCLUDE** dan **{int(label_counts.get('EXCLUDE',0))} EXCLUDE**. Sebanyak **{len(review_candidates)} record** ditandai untuk tinjauan lanjut; semua masih memiliki kelas biner. Label awal bukan ground truth independen.\n\n## 4.4 Modeling\nData latih berisi **{len(train_df)} nama unik** dan data uji **{len(test_df)} nama unik**, dengan tidak ada nama identik yang berpindah antar subset. TF-IDF karakter 3-5 gram dan tiga model digunakan: Logistic Regression, Linear SVM, Multinomial Naive Bayes. Pengujian silang dilakukan 3-fold pada data latih.\n\n## 4.5 Evaluation\n**Status evaluasi: {evaluation_status}**.\n\nModel | Accuracy | Balanced Accuracy | Precision INCLUDE | Recall INCLUDE | F1 INCLUDE | F1 Macro\n---|---:|---:|---:|---:|---:|---:\n'''+ '\n'.join(metlines)+f'''\n\n**Catatan metodologis:** Jika status masih KESESUAIAN_LABEL_ATURAN_BELUM_GROUND_TRUTH, angka tersebut hanya mengukur kesesuaian model dengan label otomatis, bukan akurasi terhadap keputusan supervisor. Jangan dijadikan klaim efektivitas final.\n\n## 4.6 Deployment\nKetiga model digunakan untuk menghasilkan rekomendasi pada **{len(result):,} record**, dengan rincian dashboard **{int(counts.get('INCLUDE',0))} INCLUDE**, **{int(counts.get('EXCLUDE',0))} EXCLUDE**, dan **{int(counts.get('REVIEW',0))} REVIEW**. Prediksi dan data peninjauan disimpan untuk diakses dari aplikasi dashboard Streamlit. Tidak ada perubahan langsung pada aturan GLPI atau XML Dictionary.\n'''
(FOLDERS['narasi']/'Draf_BAB_IV.md').write_text(report,encoding='utf-8')
# Simpan arsip hasil dalam folder proyek yang sama.
archive=BASE/'Hasil_RawData_CRISP_DM.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for fp in HASIL.rglob('*'):
        if fp.is_file():zf.write(fp,fp.relative_to(BASE))
print('Beres! Total berkas Hasil:',sum(x.is_file() for x in HASIL.rglob('*')))
print('Arsip:',archive)

---

### Catatan untuk BAB IV

1. **Label otomatis berasal dari daftar 9 keluarga produk IT; bukan validasi manual per baris.**
2. Pada penelitian final, supervisor dapat mengisi `label_verifikasi` pada file `Template_Validasi_Pak_Widi.xlsx`, menyimpannya ke folder proyek sebagai `Label_Terverifikasi.xlsx`, dan menjalankan ulang notebook.
3. Jika tujuanmu membandingkan model dengan aturan sederhana, gunakan **data uji yang benar-benar diverifikasi**. Evaluasi label aturan terhadap aturan pembuatnya sendiri akan menghasilkan kesimpulan melingkar.
4. Output `EXCLUDE` pada aplikasi merupakan **rekomendasi pemfilteran**, bukan perintah otomatis untuk menghapus software atau membuat aturan di GLPI.